# ASSIGNMENT GenAI – Prompt Engineering (LangChain Prompt Templates)

## Mini Prompt Engine for Dynamic LLM Prompting

---

## Objective

Design and implement a **Mini Prompt Engine** that converts raw user inputs into structured, reusable prompts using **LangChain PromptTemplate and ChatPromptTemplate** instead of hardcoded strings.  
The engine focuses on modularity, input validation, and role-based prompting suitable for real-world LLM pipelines.

---

## Tools and Technologies

- **Language:** Python  
- **Frameworks/Libraries:**
  - LangChain Core (`PromptTemplate`, `ChatPromptTemplate`)
  - Standard Python (`pprint`, `typing`)  
- **Environment:**  
  - Jupyter Notebook / Google Colab  
- **Design Principles:**  
  - No f-strings or hardcoded prompts in the core logic  
  - Clear separation between:
    - Template definitions
    - Validation logic
    - Prompt generation

---

## Dataset / Inputs

- `topic` – subject being explained (e.g., "Neural Networks", "Machine Learning")  
- `audience` – level or type of reader (e.g., "beginner", "intermediate", "expert")  
- `tone` – style of communication (e.g., "formal", "casual", "fun")  
- `role` – behavioral role in chat mode (e.g., "teacher", "interviewer", "motivator")  
- `style` – prompt style (e.g., "teaching", "interview", "storytelling")

These inputs are passed through a **validation layer** and then into **prompt templates** for robust, reusable prompt generation.

---

## Model / Prompting Setup

- **Core Abstractions:**
  - `PromptTemplate` – for single-turn, text-based templates with `input_variables`
  - `ChatPromptTemplate` – for multi-turn, role-based chat prompts (`system` + `human` messages)

- **Prompt Design Characteristics:**
  - Templates are defined **once** and reused across tasks
  - No concatenation via f-strings in the main logic
  - Behavior and style are controlled only through:
    - Template text
    - Input variables
    - Role configuration

---

## Pipeline Design

### High-Level Flow

```text
User Input
   ↓
Validation Layer (audience, tone, style)
   ↓
Prompt Template Selection (by style / role)
   ↓
Dynamic Prompt Generation (format with variables)
   ↓
LLM-Ready Prompt String (or chat message list)
```

### Core Components

1. **Validation Layer**
   - Ensures `audience` and `tone` belong to allowed sets
   - Optionally:
     - Raises errors, or
     - Auto-corrects to default values

2. **Template Registry**
   - A dictionary of **named prompt templates**:
     - Teaching
     - Interview
     - Storytelling
   - Enables easy extension by adding new styles without changing core logic

3. **Chat Role Behaviors**
   - Role → behavior mapping:
     - `teacher` → explain clearly with examples
     - `interviewer` → ask relevant questions
     - `motivator` → encourage and inspire
   - Used with `ChatPromptTemplate.from_messages`

4. **Mini Prompt Engine**
   - A high-level function that:
     - Accepts simple inputs
     - Validates them
     - Selects an appropriate template
     - Returns a fully formatted, LLM-ready prompt

---

## Inference and Practical Usage

A typical **inference-time usage** pattern for this mini prompt engine:

1. Collect end-user preferences:
   - Topic: `"Transformers"`
   - Audience: `"intermediate"`
   - Tone: `"casual"`
   - Style: `"storytelling"`

2. Pass to the engine:

```python
user_result = mini_prompt_engine(
    topic="Transformers",
    audience="intermediate",
    tone="casual",
    style="storytelling"
)
```

3. Take `user_result["generated_prompt"]` and feed it into any LLM (e.g., OpenAI, local model).  
4. Optionally, use `build_chat_prompt(role, topic)` when working with **chat-based** LLMs requiring system + user messages.

This design cleanly separates **prompt construction** from **model inference**, making the system modular, reusable, and easy to extend.

---

## Final Insight

This assignment transforms prompt writing into **prompt system design**:

- Instead of crafting a new hardcoded prompt every time, we:
  - Define reusable templates
  - Validate user inputs
  - Switch styles and roles via configuration
- This approach scales to real-world Generative AI apps where prompts must be:
  - Consistent
  - Maintainable
  - Dynamic
  - Easy to audit and extend

The resulting Mini Prompt Engine can be plugged into any LLM backend while keeping the **prompt logic centralized, testable, and reusable**.

In [21]:
!pip install -qU langchain langchain-core langchain-community langchain-groq langsmith langchain-huggingface langchain-chroma sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 372.7/372.7 kB 10.9 MB/s eta 0:00:00


In [22]:
import os
import getpass

# Groq API key
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")

# LangSmith API key
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LANGSMITH API key: ")

# Enable LangSmith tracing
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "langchain-blog-groq-demo"

In [23]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [35]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=800
)

In [27]:
response = llm.invoke("In exactly one sentence, explain why LangChain is useful.")
print(response.content)

LangChain is a Python library that enables developers to build conversational AI applications by providing a framework for chaining together multiple AI models and data sources to create more complex and context-aware interactions.


In [30]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI educator."),
    ("human", "Explain {topic} in simple technical language with one practical example.")
])

In [31]:
formatted_prompt = prompt.invoke({"topic": "LangChain prompt templates"})

In [32]:
print("---- FORMATTED PROMPT ----")
for msg in formatted_prompt.to_messages():
    print(f"Role: {msg.type}")
    print(f"Content:\n{msg.content}")
    print("--------------")

---- FORMATTED PROMPT ----
Role: system
Content:
You are an expert AI educator.
--------------
Role: human
Content:
Explain LangChain prompt templates in simple technical language with one practical example.
--------------


In [36]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
chain = prompt | llm | StrOutputParser()

In [37]:
answer = chain.invoke({"topic": "LangChain prompt templates"})

print("---- MODEL ANSWER ----")
print(answer)

---- MODEL ANSWER ----
**Introduction to LangChain Prompt Templates**

LangChain is a Python library that enables developers to build conversational AI applications. One of its key features is the ability to create and manage prompt templates. Prompt templates are pre-defined structures that guide the conversation flow and help the AI model generate more accurate and relevant responses.

**What are LangChain Prompt Templates?**

LangChain prompt templates are essentially placeholders for user input or context that can be filled in with actual data. These templates are designed to be flexible and reusable, allowing developers to create a wide range of conversational flows.

**Key Components of LangChain Prompt Templates**

1. **Slots**: These are placeholders for user input or context. Slots can be filled in with actual data, such as names, dates, or locations.
2. **Prompt**: This is the text that is used to guide the conversation flow. Prompts can be simple or complex, depending on the

In [38]:
qa_prompt = ChatPromptTemplate.from_template(
    "Answer the following question in 3 clear bullet points:\n\nQuestion: {question}"
)

qa_chain = qa_prompt | llm | StrOutputParser()

qa_answer = qa_chain.invoke({
    "question": "Why are chains important in LangChain?"
})

print("---- Q&A CHAIN ANSWER ----")
print(qa_answer)

---- Q&A CHAIN ANSWER ----
Here are three clear bullet points explaining the importance of chains in LangChain:

• **Modularization**: Chains in LangChain allow for modularization of tasks, enabling developers to break down complex tasks into smaller, manageable components. This modularity makes it easier to reuse and combine different components to create more complex workflows.

• **Flexibility and Customization**: Chains provide a flexible framework for customizing workflows. By combining different components, developers can create tailored workflows that meet specific requirements, making LangChain a highly adaptable and versatile tool.

• **Reusability and Composition**: Chains enable reusability and composition of components, allowing developers to create new workflows by combining existing ones. This feature promotes code reuse, reduces development time, and makes it easier to maintain and update complex workflows.


In [40]:
!pip install -qU "langchain[classic]"

In [41]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

memory = ConversationBufferMemory(return_messages=True)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

resp1 = conversation.predict(input="Hi, my name is Paritosh and I am learning LangChain.")
print("Response 1:", resp1)

resp2 = conversation.predict(input="What did I just tell you about myself?")
print("Response 2:", resp2)

/tmp/ipykernel_6326/1906177233.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True)
/tmp/ipykernel_6326/1906177233.py:6: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: Hi, my name is Paritosh and I am learning LangChain.
AI:

> Finished chain.
Response 1: Nice to meet you, Paritosh. LangChain is an open-source Python library for building conversational AI applications. It was created by Joshua Browning and is primarily used for building chatbots, virtual assistants, and other conversational interfaces. LangChain is designed to be highly customizable and flexible, allowing developers to easily integrate it with various natural language processing (NLP) and machine learning models. What specifically are you trying to learn about LangChain, Paritosh?


> Entering new ConversationChain chain...
Prompt after formatting:
The foll

In [43]:
from langchain.tools import tool

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integers and return the product."""
    return a * b

print("Tool test:", multiply_numbers.invoke({"a": 7, "b": 9}))

Tool test: 63


In [45]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integers and return the product."""
    return a * b

tools = [multiply_numbers]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools whenever calculation is needed."
)

In [47]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 23 multiplied by 17? Then explain the operation used."}
    ]
})

print("---- FINAL ANSWER ----")
print(result["messages"][-1].content)

---- FINAL ANSWER ----
The operation used is multiplication, which is a basic arithmetic operation that combines two numbers to produce a product. In this case, we are multiplying 23 by 17 to get the product 391.

Multiplication can be thought of as repeated addition. For example, 23 multiplied by 17 can be thought of as adding 23 together 17 times:

23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 + 23 = 391

This is why the result of multiplying 23 by 17 is 391.


In [48]:
sample_text = """
LangChain is a framework for building LLM-powered applications.
Its key components include models, prompts, chains, memory, agents, tools, document loaders, and vector stores.
Prompt templates help structure model input.
Agents can dynamically decide which tools to use.
Vector stores allow semantic retrieval over embedded documents.
"""

with open("langchain_notes.txt", "w") as f:
    f.write(sample_text)

print("Sample document created successfully.")

Sample document created successfully.


In [49]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("langchain_notes.txt")
docs = loader.load()

print("Number of documents loaded:", len(docs))
print(docs[0].page_content)

Number of documents loaded: 1

LangChain is a framework for building LLM-powered applications.
Its key components include models, prompts, chains, memory, agents, tools, document loaders, and vector stores.
Prompt templates help structure model input.
Agents can dynamically decide which tools to use.
Vector stores allow semantic retrieval over embedded documents.



In [50]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model initialized.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialized.


In [51]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="langchain-demo"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Vector store and retriever created.")

Vector store and retriever created.


In [52]:
query = "What are vector stores used for in LangChain?"
retrieved_docs = retriever.invoke(query)

print("Query:", query)
print("---- Retrieved Documents ----")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}:")
    print(doc.page_content)
    print("--------------")

Query: What are vector stores used for in LangChain?
---- Retrieved Documents ----
Document 1:

LangChain is a framework for building LLM-powered applications.
Its key components include models, prompts, chains, memory, agents, tools, document loaders, and vector stores.
Prompt templates help structure model input.
Agents can dynamically decide which tools to use.
Vector stores allow semantic retrieval over embedded documents.

--------------


In [53]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Use the context below to answer the question.
If the answer is not present in the context, say you do not know.

Context:
{context}

Question:
{question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

rag_question = "What are the core components of LangChain?"
rag_answer = rag_chain.invoke(rag_question)

print("Question:", rag_question)
print("---- RAG ANSWER ----")
print(rag_answer)

Question: What are the core components of LangChain?
---- RAG ANSWER ----
The core components of LangChain include:

1. Models
2. Prompts
3. Chains
4. Memory
5. Agents
6. Tools
7. Document loaders
8. Vector stores


## Conclusion

This assignment demonstrated how LangChain PromptTemplate and ChatPromptTemplate can be used to build reusable, dynamic, and structured prompt systems. Instead of hardcoding prompts, templates were designed in a modular way, combined with validation logic, and tested across multiple use cases.